# Random Forest on Morgan Fingerprints (Representation B)

This notebook trains and evaluates a **Random Forest classifier** using **Morgan fingerprints** as the molecular representation.



### Import Required Libraries

- **numpy**: Load fingerprint matrices and labels
- **sklearn (RandomForest, KFold, metrics)**: Model and evaluation
- **warnings**: Suppress noisy warnings


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import KFold

# ignore warnings
import warnings
warnings.filterwarnings("ignore")

seed = 20231124
np.random.seed(seed)


## Step 1: Load Fingerprint Data

Load the fingerprint matrix and labels generated in `1_get_fingerprints.ipynb`:
- `training_fingerprints_matrix.npy`: \(X\), fingerprint features
- `training_fingerprints_labels.npy`: \(y\), activity labels (0/1)


In [ ]:
# Load fingerprint matrix (X) and labels (y)
X_fp = np.load("training_fingerprints_matrix.npy")
y_fp = np.load("training_fingerprints_labels.npy")

print(f"Fingerprint matrix shape: {X_fp.shape}")
print(f"Labels shape: {y_fp.shape}")
print(f"Class balance: {(y_fp==1).sum()} active / {(y_fp==0).sum()} inactive")


## Step 2: Random Forest with 10-Fold Cross-Validation

We use the **same evaluation strategy** as for descriptors:
- **Model**: `RandomForestClassifier`
- **CV**: 10-fold cross-validation
- **Metrics**:
  - **Accuracy** per fold + mean accuracy
  - **ROC AUC** per fold + mean AUC

This allows a fair comparison between descriptors and fingerprints.


In [ ]:
# 10-fold cross-validation on fingerprints
kf_fp = KFold(n_splits=10, shuffle=True, random_state=seed)

rf_acc_fp = []
rf_auc_fp = []

fold = 0
for train_index, test_index in kf_fp.split(X_fp):
    X_train_fp, X_test_fp = X_fp[train_index], X_fp[test_index]
    y_train_fp, y_test_fp = y_fp[train_index], y_fp[test_index]

    model_fp = RandomForestClassifier(random_state=42)
    model_fp.fit(X_train_fp, y_train_fp)

    # Predictions
    preds_fp = model_fp.predict(X_test_fp)
    proba_fp = model_fp.predict_proba(X_test_fp)[:, 1]

    # Metrics
    acc_fp = accuracy_score(y_test_fp, preds_fp)
    auc_fp = roc_auc_score(y_test_fp, proba_fp)

    rf_acc_fp.append(acc_fp)
    rf_auc_fp.append(auc_fp)

    fold += 1
    print(f"Done with fold {fold}/10")

print("\nRandom Forest on Fingerprints (Representation B)")
print("Accuracy per fold:", rf_acc_fp)
print("Mean CV accuracy:", sum(rf_acc_fp) / len(rf_acc_fp))
print("AUC per fold:", rf_auc_fp)
print("Mean CV AUC:", sum(rf_auc_fp) / len(rf_auc_fp))
